# Computational Complexity of Implemented Functions

This document summarizes the time complexity (Big-O) of the main classes and methods implemented in the module. The objective is to identify whether any of the algorithms have a complexity greater than O(1), and to document only those whose complexity grows with the size of the data set.

In general, most methods operate linearly on the data, which is appropriate for tasks such as analysis, cleaning, and transformation of information. There are no methods with quadratic or exponential complexity.

The functions that do have complexity greater than constant are detailed below.

_____________________________________________________________________________________________________________________

# preprocessing.py Big-O -----------------------------------

In [ ]:
#-----------------------------------------
# Data Loading Class
#-----------------------------------------

class DataLoad:
    def __init__(self, url):
        self.url = url

    def __call__(self):
        response = requests.get(self.url)
        response.raise_for_status() 
        data = response.json()
        df = pd.DataFrame(data)
        return df

#-----------------------------------------
# Data Searching and Filtering Class
#-----------------------------------------

class DataScearch:
    def __init__(self, df):
        self.df = df

    def __call__(self, column, value):
        if column not in self.df.columns:
            
           raise ValueError(f"Column '{column}' does not exist in the DataFrame")

        elif self.df[column].dtype == "O":
            
            result = self.df[self.df[column].str.contains(value)]
            
        else:
        
            result = self.df[self.df[column] == value]
    
        return result
    
    # Method to extract CO2 readings and corresponding timestamps from the filtered DataFrame
    def __co2read__(self, result):

        if result.empty:
    
            raise ValueError("The filtered DataFrame is empty")
        
        co2_array = result["co2readings"].dropna().to_numpy()
        co2_array = np.concatenate(co2_array)
        
        start_time = datetime.fromisoformat(result.iloc[0]["startOfMeasurement"])  
        interval = int(result.iloc[0]["interval"])
        time_array = np.array([start_time + timedelta(minutes=int(i * interval)) for i in range(len(co2_array))])


        return co2_array, time_array


#-----------------------------------------
# Data Cleaning Class
#-----------------------------------------

class DataCleaner:
    def __init__(self, df):
        self.df = df

    def __call__(self, method="mean"):

        # clean empty strings and common NaN representations of pandas and numpy null values
        self.df = self.df.replace(
            [r'^\s*$', 'NaN', 'nan', 'NAN', 'N/A', 'None', None],
            np.nan,
            regex=True
        )

        # Identify columns with list or dict types to exclude them from imputation
        list_cols = [
            col for col in self.df.columns 
            if self.df[col].apply(lambda x: isinstance(x, (list, dict))).any()
        ]

        # numeric columns imputation
        num_cols = self.df.select_dtypes(include='number').columns.difference(list_cols)
        if method == "mean":
            self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean())
        elif method == "median":
            self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].median())
        else:
            raise ValueError('Use mean or median')

        # string/categorical columns imputation
        cat_cols = self.df.select_dtypes(exclude=['number']).columns.difference(list_cols)
        for col in cat_cols:
            mode_series = self.df[col].mode()
            if not mode_series.empty:
                self.df[col].fillna(mode_series.iloc[0], inplace=True)

        return self.df

#-----------------------------------------
# Data Printing Class
#-----------------------------------------

class DataPrint:
    def __init__(self, df):
        self.df = df

    def __call__(self, columns):

        for col in columns:
            if col not in self.df.columns:
                print(f'The column {col} does not exist in the DataFrame.\n')
                continue

            counts = self.df[col].value_counts(dropna=False)
            total = len(self.df[col])
            print(f'\n{counts}')
            print(f'Size: {total}\n')

#-----------------------------------------
# CO2 Classification Class
#-----------------------------------------

class ClassifierCO2:
    def __init__(self):
        pass

    @staticmethod

    def classify_co2(ppm):
        if ppm < 1000:
            return "Safe"
        elif ppm < 2000:
            return "Moderate"
        elif ppm < 5000:
            return "High"
        elif ppm < 10000:
            return "Risky"
        elif ppm < 15000:
            return "Dangerous"
        elif ppm < 30000:
            return "Severe"
        elif ppm < 50000:
            return "Critical"
        else:
            return "Lethal"

#-----------------------------------------
# Ventilation Classification Class
#-----------------------------------------

class VentilationClassifier:
    def __init__(self):
        pass
    
    @staticmethod

    def classify_ventilation(row):
        mech = str(row["ventilationSystem"]).strip().lower() == "true"
        nat = str(row["windowsOpen"]).strip().lower() == "true"
        
        if mech and nat:
            return "both"
        elif mech:
            return "mechanical"
        elif nat:
            return "natural"
        else:
            return "none"

#-----------------------------------------
# Time of Day Classification Class
#-----------------------------------------

class TimeOfDayClassifier:
    
    def __init__(self):
        pass

    @staticmethod
    def classify_hour(dt):

        hour = dt.hour
        if 21 <= hour or hour < 3:
            return "Midnight"
        elif 3 <= hour < 9:
            return "Morning"
        elif 9 <= hour < 15:
            return "Noon"
        elif 15 <= hour < 21:
            return "Afternoon"
        else:
            return np.nan

    def classify_list(self, time_list):

        if not isinstance(time_list, list) or len(time_list) == 0:
            return np.nan
        try:
            t = pd.to_datetime(time_list)
            return self.classify_hour(t[0])
        except Exception:
            return np.nan

#-----------------------------------------
# Time Series Construction Class
#-----------------------------------------

class TimeSeries:
    def __init__(self, df):
        self.df = df

    def __call__(self, readings_col, start_col, interval_col, absolute=True): #True absolute time, False relative time

        def build_times(row):
            readings = row[readings_col]
            interval = row[interval_col]
            if not isinstance(readings, list) or len(readings) == 0:
                return np.nan
            if pd.isna(interval) or interval <= 0:
                return np.nan
            
            dt = interval * 60.0 
            rel_times = np.arange(0, len(readings) * dt, dt)
            
            if absolute:
                start_time = pd.to_datetime(row[start_col])
                abs_times = [start_time + pd.to_timedelta(t, unit='s') for t in rel_times]
                return abs_times
            else:
                return list(rel_times)

        self.df["timelist"] = self.df.apply(build_times, axis=1)
        return self.df

#-----------------------------------------
# Class to convert time to seconds
#-----------------------------------------

class TimeSeconds:
    def __init__(self, time_array):
        self.time_array = time_array
    
    def __call__(self):
        delta = [t - self.time_array[0] for t in self.time_array]
        time_seconds = np.array([d.total_seconds() for d in delta])
        return time_seconds  



### Methods with O(n) Complexity

The following methods traverse the entire DataFrame, lists, or arrays, so their complexity is linear with respect to the number of elements n:

``DataLoad.__call__()``
Converts the downloaded JSON into a DataFrame by traversing all records.

``DataSearch.__call__()``
Filters the DataFrame using equality or str.contains().

``DataSearch.__co2read__()``
Cleans, concatenates, and reconstructs the CO₂ time series array.

``DataCleaner.__call__()``
Includes replacements, imputation, scanning columns with lists and dictionaries, all of which are linear operations.

``DataPrint.__call__()``
Uses value_counts(), which requires traversing the entire column.

``TimeOfDayClassifier.classify_list()``
Conversion of timestamp lists to datetime objects.

``TimeSeries.__call__()``
Construction of absolute or relative time lists per row.

``TimeSeconds.__call__()``
Time differences transformed to seconds.

### Methods with O(1) Complexity

These methods only involve direct comparisons and simple conditional logic:

``ClassifierCO2.classify_co2()``

``VentilationClassifier.classify_ventilation()``

``TimeOfDayClassifier.classify_hour()``

### Summary Table
| Method                                 | Complexity | Description                                           |
| -------------------------------------- | ----------- | ----------------------------------------------------- |
| `DataLoad.__call__()`                  | O(n)        | Builds the DataFrame from the JSON                    |
| `DataScearch.__call__()`               | O(n)        | Filtering by equality or partial match                |
| `DataScearch.__co2read__()`            | O(n)        | Builds the CO₂ array + timestamps                     |
| `DataCleaner.__call__()`               | O(n)        | Cleaning, replacement, and imputation                 |
| `DataPrint.__call__()`                 | O(n)        | Generation of counts                                   |
| `TimeOfDayClassifier.classify_list()`  | O(n)        | Processing of temporal lists                          |
| `TimeSeries.__call__()`                | O(n)        | Construction of time series                           |
| `TimeSeconds.__call__()`               | O(n)        | Conversion of timestamps to seconds                   |
| *Basic classifiers*                    | O(1)        | Conditionals and comparisons                          |


### Conclusion

The analysis shows that the most relevant functions of the process work linearly O(n), which is suitable for manipulating tabular data and time series.
No method with a complexity greater than linear was found, and simple classifiers maintain a constant complexity O(1).
Overall, the module is efficient and scalable for exploratory data analysis (EDA), processing, and sequential transformation.

# analysis.py Big-O ------------------------------------------

In [ ]:
#-----------------------------------------
# Feature Statistics Extraction Class
#-----------------------------------------

class FeatureExtractor:
    def __init__(self, df, list_col):
        self.df = df
        self.list_col = list_col

    def __call__(self, prefix=None):

        if prefix is None:
            prefix = self.list_col

        self.df[f"{prefix}Med"] = self.df[self.list_col].apply(np.median)
        self.df[f"{prefix}Std"] = self.df[self.list_col].apply(np.std)
        self.df[f"{prefix}Var"] = self.df[self.list_col].apply(np.var)
        self.df[f"{prefix}Min"] = self.df[self.list_col].apply(np.min)
        self.df[f"{prefix}Max"] = self.df[self.list_col].apply(np.max)
        self.df[f"{prefix}Range"] = self.df[self.list_col].apply(lambda x: np.max(x) - np.min(x))
        self.df[f"{prefix}RMS"] = self.df[self.list_col].apply(lambda x: np.sqrt(np.mean(np.square(x))))
        self.df[f"{prefix}CV"] = self.df[f"{prefix}Std"] / self.df[f"{prefix}Med"]

        return self.df

#-----------------------------------------
# Temporal Feature Statistics Extraction Class
#-----------------------------------------

class TemporalFeatureExtractor:

    def __init__(self, df, co2_col, time_col):
        self.df = df
        self.co2_col = co2_col
        self.time_col = time_col

    def convert_s(self, time_list):
        if not isinstance(time_list, list) or len(time_list) < 2:
            return None
        try:
            t = pd.to_datetime(time_list)
            return np.array([(ti - t[0]).total_seconds() for ti in t])
        except Exception:
            return None

    def __call__(self):
        def safe_polyfit(t, y, deg):
            try:
                if t is None or len(t) < 3:
                    return np.nan
                return np.polyfit(t, y, deg)[0]
            except Exception:
                return np.nan

        self.df["co2readingsSlope"] = self.df.apply(
            lambda r: safe_polyfit(self.convert_s(r[self.time_col]), r[self.co2_col], 1),
            axis=1
        )

        self.df["co2readingsCurvature"] = self.df.apply(
            lambda r: safe_polyfit(self.convert_s(r[self.time_col]), r[self.co2_col], 2),
            axis=1
        )

        self.df["co2readingsMean_diff"] = self.df[self.co2_col].apply(
            lambda x: np.mean(np.abs(np.diff(x))) if isinstance(x, list) and len(x) > 1 else np.nan
        )

        self.df["co2readingsStd_diff"] = self.df[self.co2_col].apply(
            lambda x: np.std(np.diff(x)) if isinstance(x, list) and len(x) > 1 else np.nan
        )

        return self.df

#-----------------------------------------
# Class for extracting the derivative and second derivative
#-----------------------------------------

class Derivative:
    def __init__(self, x, t):
        self.x = x
        self.t = t
    
    def __call__(self):
        #diff is the discrete difference for the forward derivative
        dx = np.diff(self.x) 
        dt = np.diff(self.t)
        derivative = dx / dt
        return derivative

class SecondDerivative(Derivative):
    def __call__(self):
        first_derivative = super().__call__()
        dx2 = np.diff(first_derivative)
        dt = np.diff(self.t)[1:]  # Second derivative fit [All except the first]
        second_derivative = dx2 / dt
        return second_derivative

#-----------------------------------------
# Kolmogorov–Smirnov Test
#-----------------------------------------

class KolmogorovSmirnovTest:

    def __init__(self, data):
        self.data = np.array(data)

    def __call__(self):
        data = self.data[~np.isnan(self.data)]
        if len(data) < 3:
            return {"test": "KS", "error": "Insufficient data"}

        mean = np.mean(data)
        std = np.std(data)

        ref = np.random.normal(mean, std, size=len(data))

        stat, p = ks_2samp(data, ref)

        return {"test": "KS", "statistic": stat, "pvalue": p}


#-----------------------------------------
# Shapiro–Wilk Test
#-----------------------------------------

class ShapiroWilkTest:
    def __init__(self, data):
        self.data = np.array(data)

    def __call__(self):
        data = self.data[~np.isnan(self.data)]
        if len(data) < 3:
            return {"test": "Shapiro", "error": "Insufficient data"}

        stat, p = shapiro(data)
        return {"test": "Shapiro", "statistic": stat, "pvalue": p}


#-----------------------------------------
# Kruskal–Wallis
#-----------------------------------------

class KruskalWallisTest:
    def __init__(self, groups: dict):
        self.groups = groups

    def __call__(self):
        data = [
            np.array(values)[~np.isnan(values)]
            for values in self.groups.values()
        ]
        data = [g for g in data if len(g) > 1]
        if len(data) < 2:
            return {"test": "Kruskal-Wallis", "statistic": np.nan, "pvalue": np.nan}
        try:
            stat, p = kruskal(*data)
        except Exception:
            stat, p = np.nan, np.nan
        return {"test": "Kruskal-Wallis", "statistic": stat, "pvalue": p}

#-----------------------------------------
# Chi-Cuadrado
#-----------------------------------------

class ChiSquareTest:

    def __init__(self, contingency):
        self.contingency = contingency

    def __call__(self):
        stat, p, dof, expected = chi2_contingency(self.contingency)
        return {
            "test": "Chi-Square",
            "statistic": stat,
            "pvalue": p,
            "dof": dof,
            "expected": expected
        }

### Methods with O(n) Complexity

The following methods iterate through full DataFrame columns, lists, or arrays.
Their complexity is linear with respect to the number of elements n:

``FeatureExtractor.__call__()``
Computes multiple statistical features (median, std, var, min, max, range, RMS, CV) by applying functions to full lists.

``TemporalFeatureExtractor.__call__()``
Processes CO₂ and timestamp lists to compute slope, curvature, and differences, using per-row operations that traverse full arrays.

``TemporalFeatureExtractor.convert_s()``
Transforms timestamp lists into seconds relative to the first element.

``Derivative.__call__()``
Computes the numerical first derivative using np.diff(), a linear pass.

``SecondDerivative.__call__()``
Computes the second derivative by applying np.diff() again.

``KolmogorovSmirnovTest.__call__()``
Removes NaNs, constructs a normal reference sample, and applies the KS test.

``ShapiroWilkTest.__call__()``
Filters valid data and computes the Shapiro–Wilk statistic.

``KruskalWallisTest.__call__()``
Cleans groups, removes NaNs, and performs the Kruskal–Wallis test.

``ChiSquareTest.__call__()``
Runs chi2_contingency() over the full contingency table.

### Methods with O(1) Complexity

These methods only perform direct arithmetic or index differences:

``np.diff()`` per element (inside Derivative / SecondDerivative)
Though used inside O(n) workflows, each operation itself is constant time.

``safe_polyfit()`` on very small arrays
Called for each row, but the polynomial fit itself has constant cost given fixed degree (1 or 2).

### Summary Table
| Method                                | Complexity | Description                                            |
| ------------------------------------- | ---------- | ------------------------------------------------------ |
| `FeatureExtractor.__call__()`         | O(n)       | Computes statistical features (median, std, var, etc.) |
| `TemporalFeatureExtractor.__call__()` | O(n)       | Extracts temporal features (slope, curvature, diffs)   |
| `Derivative.__call__()`               | O(n)       | Computes the first numerical derivative                |
| `SecondDerivative.__call__()`         | O(n)       | Computes the second numerical derivative               |
| `KolmogorovSmirnovTest.__call__()`    | O(n)       | Performs the KS normality comparison test              |
| `ShapiroWilkTest.__call__()`          | O(n)       | Performs the Shapiro–Wilk normality test               |
| `KruskalWallisTest.__call__()`        | O(n)       | Performs the Kruskal–Wallis test for multiple groups   |
| `ChiSquareTest.__call__()`            | O(n)       | Performs the Chi-Square independence test              |
| *Basic internal operations*           | O(1)       | Conditional checks, validations, simple arithmetic     |

### Conclusion

The analysis shows that the statistical processing module operates efficiently, with all major methods exhibiting linear complexity O(n). This makes the system suitable for large DataFrames and multiple time-series lists without significant performance degradation.

No method exceeds linear complexity, and operations that involve polynomial fitting or numerical differentiation are constant-time per row, keeping overall complexity stable.

Overall, the analysis.py module is scalable, robust, and optimized for exploratory data analysis, particularly when dealing with sensor data, lists of CO₂ readings, timestamps, or grouped statistical comparisons.




# visualization.py Big-O ----------------------------------

In [ ]:
#-----------------------------------------
# Plot Builder
#-----------------------------------------

class PlotBuilder:
    def __init__(self, df=None):
        self.df = df
        self.palette1 = "light:#5A9"
        self.palette2 = "Set2"

    def _random_color(self):
        return random.choice(sns.color_palette("tab10", 10))

    #-------------------------
    # Line Plot
    #-------------------------
    def line(self, x, y, title=None):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(self.df[x], self.df[y], color=self._random_color())
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Histogram
    #-------------------------
    def hist(self, column, bins=30, title=None):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.hist(self.df[column], bins=bins, color=self._random_color())
        ax.set_xlabel(column)
        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Boxplot
    #-------------------------
    def box(self, column, by=None, title=None):
        fig, ax = plt.subplots(figsize=(10, 6))
        if by:
            sns.boxplot(x=self.df[by], y=self.df[column], ax=ax, palette=self.palette1)
        else:
            ax.boxplot(self.df[column].dropna())
        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Violin Plot
    #-------------------------
    def violin(self, column, by=None, title=None):
        fig, ax = plt.subplots(figsize=(10, 6))
        if by:
            sns.violinplot(x=self.df[by], y=self.df[column], ax=ax, palette=self.palette2)
        else:
            sns.violinplot(y=self.df[column], ax=ax, palette=self.palette)
        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Scatter Plot
    #-------------------------
    def scatter(self, x, y, title=None):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(self.df[x], self.df[y], color=self._random_color())
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Bar Plot
    #-------------------------
    def bar(self, x, y, title=None, horizontal=False):
        fig, ax = plt.subplots(figsize=(10, 6))

        if horizontal:
            ax.barh(self.df[x], self.df[y], color=self._random_color())
            ax.set_ylabel(x)
            ax.set_xlabel(y)
        else:
            ax.bar(self.df[x], self.df[y], color=self._random_color())
            ax.set_xlabel(x)
            ax.set_ylabel(y)

        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Heatmap
    #-------------------------
    def heatmap(self, title=None, annot=False):
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(self.df, cmap='viridis', annot=annot, ax=ax)
        if title:
            ax.set_title(title)
        return fig

    #-------------------------
    # Pairplot
    #-------------------------
    def pairplot(self, hue=None, diag_kind="kde", title=None):
        grid = sns.pairplot(
            self.df,
            hue=hue,
            diag_kind=diag_kind,
            palette=self.palette2,
            corner=True,
            plot_kws={"alpha": 0.6, "s": 40}
        )

        if title:
            grid.fig.suptitle(title, y=1.02)

        return grid.fig

    #-------------------------
    # Multi-Boxplot (subplot)
    #-------------------------
    def multi_boxplot(self, columns, by, title=None):
        n = len(columns)
        rows = (n // 3) + (1 if n % 3 != 0 else 0)
        fig, axes = plt.subplots(rows, 3, figsize=(18, 5 * rows))
        axes = axes.flatten()
        for ax, col in zip(axes, columns):
            sns.boxplot(
                data=self.df,
                x=by,
                y=col,
                palette=self.palette1,
                showfliers=False,
                ax=ax
            )
            ax.set_title(f"{col} by {by}", fontsize=11, fontweight="bold")
            ax.tick_params(axis="x", rotation=25)
        for i in range(len(columns), len(axes)):
            axes[i].set_visible(False)
        if title:
            fig.suptitle(title, fontsize=15, fontweight="bold", y=1.02)
        fig.tight_layout()
        return fig

#-----------------------------------------
# Export Figures
#-----------------------------------------

class FigureExporter:
    def __init__(self, fig):
        self.fig = fig

    def save(self, path, dpi=300):
        self.fig.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close(self.fig)
        return path


#-----------------------------------------
# Exxport DataFrames or dicts as cvs or png
#-----------------------------------------

class TableExporter:
    def __init__(self, table):
        if isinstance(table, pd.DataFrame):
            self.table = table
        else:
            self.table = pd.DataFrame(table)

    def to_csv(self, folder_path, filename):
        os.makedirs(folder_path, exist_ok=True)
        fullpath = os.path.join(folder_path, filename)
        self.table.to_csv(fullpath, index=True)
        return fullpath

    def to_png(self, path, dpi=300):
        fig, ax = plt.subplots(figsize=(12, 2 + 0.3 * len(self.table)))
        ax.axis("off")
        table = ax.table(
            cellText=self.table.values,
            colLabels=self.table.columns,
            loc="center"
        )
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)

        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return path

### Methods with O(n) Complexity

The following methods iterate through full DataFrame columns, compute plots by traversing arrays, or render multiple subplots.
Their complexity is linear with respect to the number of elements **n**:

`PlotBuilder.line()`
Reads full `x` and `y` columns to generate a line plot.

`PlotBuilder.hist()`
Traverses the full column to generate histogram bin counts.

`PlotBuilder.box()`
When grouping with `by`, seaborn iterates through subsets of size n.

`PlotBuilder.violin()`
Seaborn computes KDE per group, requiring linear traversal.

`PlotBuilder.scatter()`
Plots all points by iterating through full columns `x` and `y`.

`PlotBuilder.bar()`
Draws bars for each row in the DataFrame, iterating linearly.

`PlotBuilder.heatmap()`
Traverses the full matrix (n rows × m columns).

`PlotBuilder.pairplot()`
Builds all pairwise relationships, iterating through each column vector.

`PlotBuilder.multi_boxplot()`
Generates multiple subplots; each subplot processes a full column.

`TableExporter.to_png()`
Renders all rows of the table into a matplotlib Figure.


### Methods with O(1) Complexity

These methods perform constant operations that do not scale with input size:

`PlotBuilder._random_color()`
Chooses a random color from a fixed small palette.

`FigureExporter.save()`
Saving complexity depends only on fixed DPI and fig size.

`TableExporter.to_csv()`
File creation and folder checks are constant-time relative to metadata (not data writing).


### Summary Table

| Method                        | Complexity | Description                                                        |
| ----------------------------- | ---------- | ------------------------------------------------------------------ |
| `PlotBuilder.line()`          | O(n)       | Iterates over full columns to render a line plot                   |
| `PlotBuilder.hist()`          | O(n)       | Computes histogram bins from column values                         |
| `PlotBuilder.box()`           | O(n)       | Builds a boxplot per column or per group                           |
| `PlotBuilder.violin()`        | O(n)       | Computes KDE-based violin plot per column/group                    |
| `PlotBuilder.scatter()`       | O(n)       | Renders all points in x–y scatter form                             |
| `PlotBuilder.bar()`           | O(n)       | Draws bars for each row in the dataset                             |
| `PlotBuilder.heatmap()`       | O(n)       | Traverses the full numeric matrix to draw heatmap                  |
| `PlotBuilder.pairplot()`      | O(n)       | Plots all pairwise scatter/KDE relationships                       |
| `PlotBuilder.multi_boxplot()` | O(n)       | Generates multiple subplots; each requires reading its full column |
| `TableExporter.to_png()`      | O(n)       | Renders all rows of the DataFrame/table into an image              |
| `PlotBuilder._random_color()` | O(1)       | Picks a color randomly from a fixed palette                        |
| `FigureExporter.save()`       | O(1)       | Saves a static figure to disk                                      |
| `TableExporter.to_csv()`      | O(1)       | Creates folders and metadata (file writing itself is external)     |


### Conclusion

The visualization module in *visualization.py* relies primarily on matplotlib and seaborn routines that operate linearly with respect to the size of the DataFrame columns. Every visualization method that renders points, computes KDEs, or builds multi-column plots requires one traversal of the data, resulting in **O(n)** complexity.

Only utility functions such as random color selection or saving a figure operate in constant time (**O(1)**).

Overall, the module is efficient, predictable, and fully scalable for medium and large datasets, providing a consistent interface for plots, heatmaps, subplots, and export utilities.



# main.py Big-O ----------------------------------------------

In [ ]:
# -----------------------------------------
# Helpers for memory and time tracking
# -----------------------------------------

def get_memory_mb():
    process = psutil.Process()
    return process.memory_info().rss / (1024 * 1024)


# -----------------------------------------
# main function
# -----------------------------------------

def main():

    print("\n--------------- Indoor CO2 Map Analysis Start ---------------\n")

    t0 = time.time()
    mem0 = get_memory_mb()

    # Paths
    IMAGES_PATH = os.path.join("Results", "Figures")
    TABLES_PATH = os.path.join("Results", "Tables")
    PROCESSED_DATA_PATH = os.path.join("Data", "Processed")
    URL = "https://raw.githubusercontent.com/Yeikeer/IndoorCO2Map-Analysis/refs/heads/main/Data/Raw/indoorco2mapData.json"

    # Load Data-------
    df = DataLoad(URL)()
    
    # Cleaning-------
    df = DataCleaner(df)()

    # Time Series------- 
    ts = TimeSeries(df)
    df = ts("co2readings", "startOfMeasurement", "interval", absolute=True)

    # Features--------
    df = FeatureExtractor(df, list_col="co2readings")()
    df = TemporalFeatureExtractor(df, co2_col="co2readings", time_col="timelist")()

    # Classification--------
    # -CO2 Level Classification 
    df["co2class"] = df["co2readingsMed"].apply(ClassifierCO2.classify_co2)
    counts = df["co2class"].value_counts()
    valid_classes = counts[counts >= 10].index.tolist()
    df = df[df["co2class"].isin(valid_classes)].copy()
    # -Ventilation Classification 
    df["ventilationclass"] = df.apply(VentilationClassifier.classify_ventilation, axis=1)
    # -Time of Day Classification
    tod = TimeOfDayClassifier()
    df["timeday"] = df["timelist"].apply(tod.classify_list)

    # Visualization-------

    #-count by country
    country_counts = df["countryName"].value_counts().reset_index()
    country_counts.columns = ["countryName", "count"]
    country_counts10 = country_counts[country_counts["count"] >= 10].reset_index(drop=True)

    # Bar plot of counts by country
    pb = PlotBuilder(country_counts10)
    fig = pb.bar(
        x="countryName",
        y="count",
        title="Count by country",
        horizontal=True
    )
    FigureExporter(fig).save(os.path.join(IMAGES_PATH, "count_country.png"))

    #-Top countries by median CO2 levels
    filtered_df = df[df["countryName"].isin(country_counts10["countryName"])].copy()

    co2_avg_country = (
        filtered_df.groupby("countryName")["co2readingsAvg"]
        .mean()
        .reset_index()
        .rename(columns={"countryName": "Country", "co2readingsAvg": "Avg_CO2"})
    )

    co2_avg_country = (
        co2_avg_country.sort_values(by="Avg_CO2", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    # Bar plot of top 10 countries by median CO2 levels
    pb2 = PlotBuilder(co2_avg_country)
    fig2 = pb2.bar(
        x="Country",
        y="Avg_CO2",
        title="Top 10 Countries by Median CO2 Levels",
        horizontal=True
    )
    FigureExporter(fig2).save(os.path.join(IMAGES_PATH, "avg_countries.png"))

    #-CO2 Prom kind of ventilation
    vent_means = (
        df.groupby("ventilationclass")["co2readingsAvg"]
        .mean()
        .reset_index()
        .rename(columns={"ventilationclass": "Ventilation",
                        "co2readingsAvg": "Avg_CO2"})
        .sort_values(by="Avg_CO2", ascending=False)
        .reset_index(drop=True)
    )

    # Bar plot of counts by ventilation class
    pb3 = PlotBuilder(vent_means)
    fig3 = pb3.bar(
        x="Ventilation",
        y="Avg_CO2",
        title="Average CO2 Levels by Ventilation Class",
        horizontal=False
    )
    FigureExporter(fig3).save(os.path.join(IMAGES_PATH, "avg_ventilation.png"))

    #-CO2 Prom by time of day
    timeday_means = (
        df.groupby("timeday")["co2readingsAvg"]
        .mean()
        .reset_index()
        .rename(columns={"timeday": "TimeOfDay",
                        "co2readingsAvg": "Avg_CO2"})
        .sort_values(by="Avg_CO2", ascending=False)
        .reset_index(drop=True)
    )

    # Bar plot of counts by time of day
    pb4 = PlotBuilder(timeday_means)
    fig4 = pb4.bar(
        x="TimeOfDay",
        y="Avg_CO2",
        title="Average CO2 Levels by Time of Day",
        horizontal=False
    )
    FigureExporter(fig4).save(os.path.join(IMAGES_PATH, "avg_timeofday.png"))

    #-CO2 levels by CO2 class
    #Violin plot of CO2 levels by CO2 class
    pb4 = PlotBuilder(df)

    fig4 = pb4.violin(
        column="co2readingsAvg",   # variable numérica
        by="co2class",             # variable categórica
        title="CO2 Distribution by CO2 Class"
    )

    FigureExporter(fig4).save(os.path.join(IMAGES_PATH, "violin_co2class.png"))

    #-Correlation matrix
    corr = df[
        ["co2readingsAvg", "co2readingsMed", "co2readingsStd", "co2readingsVar",
        "co2readingsRange", "co2readingsRMS", "co2readingsCV",
        "co2readingsSlope", "co2readingsCurvature",
        "co2readingsMean_diff", "co2readingsStd_diff"]
    ].corr()

    # Heatmap of correlation matrix
    pb5 = PlotBuilder(corr)
    fig5 = pb5.heatmap(
        title="Correlation Heatmap of CO2 Metrics",
        annot=True
    )

    FigureExporter(fig5).save(os.path.join(IMAGES_PATH, "heatmap_metrics.png"))

    #-Feuture comparatives plots
    cols = df[[
        "co2class",               # Classifier
        "co2readingsMed",          # CO2 median
        "co2readingsStd",          # Standard deviation
        "co2readingsCV",           # Coefficient of Variation
        "co2readingsSlope",        # Slope
        "co2readingsCurvature",    # Curvature
        "co2readingsStd_diff",     # Variability of changes
    ]]

    # Pairplot of selected features by CO2 class
    pb6 = PlotBuilder(cols)
    fig6 = pb6.pairplot(
        hue="co2class",
        diag_kind="kde",
        title="Pairplot of CO2 Features by CO2 Class"
    )

    FigureExporter(fig6).save(os.path.join(IMAGES_PATH, "pairplot_features.png"))

    # Stadistics tests -------
    # Numeric variables of interest
    numeric_features = [
        "co2readingsMed",
        "co2readingsStd",
        "co2readingsCV",
        "co2readingsSlope",
        "co2readingsCurvature",
        "co2readingsStd_diff"
    ]

    # Categorical variables of interest
    categorical_features = [
        "timeday",
        "ventilationclass",
        "countryName",
        "osmKey",
        "co2class"   # Classifier
    ]

    #-Kolmogorov-Smirnov Test for normality
    ks_results = []
    for feature in numeric_features:
        data = df[feature].values
        test = KolmogorovSmirnovTest(data)()
        test["feature"] = feature
        ks_results.append(test)
    ks_df = pd.DataFrame(ks_results)[["feature", "test", "statistic", "pvalue"]]
    TableExporter(ks_df).to_png(os.path.join(TABLES_PATH, "kolmogorov_tests.png"))

    #-Kruskal-Wallis Test for differences between groups
    kw_results = []
    for feature in numeric_features:
        groups = df.groupby("co2class")[feature].apply(list).to_dict()
        test = KruskalWallisTest(groups)()
        test["feature"] = feature
        kw_results.append(test)
    kw_df = pd.DataFrame(kw_results)[["feature", "test", "statistic", "pvalue"]]
    TableExporter(kw_df).to_png(os.path.join(TABLES_PATH, "kruskal_tests.png"))

    #-Chi-Squared Test for independence between categorical variables
    chi_results = []
    for cat in categorical_features:
        contingency = pd.crosstab(df[cat], df["co2class"])
        test = ChiSquareTest(contingency)()
        chi_results.append({
            "variable": cat,
            "statistic": test["statistic"],
            "pvalue": test["pvalue"],
            "dof": test["dof"]
        })
    chi_df = pd.DataFrame(chi_results)
    TableExporter(chi_df).to_png(os.path.join(TABLES_PATH, "chi_square_tests.png"))

    # Visualization final -------

    #-Multi box plot
    significant_features = kw_df[kw_df["pvalue"] < 0.05]["feature"].tolist()
    pb7 = PlotBuilder(df)
    fig7 = pb7.multi_boxplot(
        columns=significant_features,
        by="co2class",
        title="Significant Features by CO2 Class"
    )

    FigureExporter(fig7).save(os.path.join(IMAGES_PATH, "significant_features.png"))

    # Export processed data -------
    TableExporter(df).to_csv(PROCESSED_DATA_PATH, "indoorco2map_processed.csv")

    # Performance ------

    t1 = time.time()
    mem1 = get_memory_mb()

    print("\n--------------- Indoor CO2 Map End ---------------\n")

    print("\n--- Performance ---")
    print(f"Total Time: {t1 - t0:.2f} s")
    print(f"Used Memory: {mem1 - mem0:.2f} MB")

# -----------------------------------------
# Entry point
# -----------------------------------------

if __name__ == "__main__":
    main()


### Segments with O(n) Complexity

These operations traverse the dataset linearly:

#### **Data loading and cleaning**  
```python
df = DataLoad(URL)()
df = DataCleaner(df)()
````

Each row is visited once. Complexity: **O(n)**

* **Time series processing**

```python
ts = TimeSeries(df)
df = ts("co2readings", "startOfMeasurement", "interval", absolute=True)
```

Each row is traversed to compute time features. **O(n)**

#### **Feature extraction**

```python
df = FeatureExtractor(df, list_col="co2readings")()
df = TemporalFeatureExtractor(df, co2_col="co2readings", time_col="timelist")()
```

Each row is processed for statistics → **O(n)**

#### **Row-wise classification**

```python
df["co2class"] = df["co2readingsMed"].apply(ClassifierCO2.classify_co2)
df["ventilationclass"] = df.apply(VentilationClassifier.classify_ventilation, axis=1)
df["timeday"] = df["timelist"].apply(tod.classify_list)
```

Each `apply` processes all rows once. **O(n)**

### Segments with O(n²) or Higher Complexity

Some loops and group operations can scale worse than linear:

#### **Kruskal-Wallis test with groupby and list conversion**

```python
for feature in numeric_features:
    groups = df.groupby("co2class")[feature].apply(list).to_dict()
    test = KruskalWallisTest(groups)()
```

* `groupby().apply(list)` iterates each row for each group.

  * If g = number of groups, worst-case cost per group traversal: O(n * g)
  * If groups are not very small relative to n, this can approach **O(n²)** in pathological cases.

#### **Chi-Square tests over categorical features**

```python
for cat in categorical_features:
    contingency = pd.crosstab(df[cat], df["co2class"])
```

* `crosstab` internally iterates rows multiple times depending on unique categories.

  * Complexity: O(n * k1 * k2), where k1/k2 = number of unique values per variable
  * For large categorical cardinality, this can be > O(n)

#### **Pairplot**

```python
pb6.pairplot(cols, hue="co2class")
```

* Iterates all pairs of columns and all rows: O(c² * n)

* If c ≈ 10 (fixed), linear in n; if c grows with n, becomes **superlinear**.


#### **Multi-boxplot over significant features**

```python
pb7.multi_boxplot(columns=significant_features, by="co2class")
```

* Traverses all rows for each feature and for each group
* Complexity: O(f * g * n), f = number of features, g = number of groups
* If f or g grows with n, this could approach **O(n²)**.


### Segments with O(1) Complexity

Operations independent of dataset size:

* `os.path.join()`, `get_memory_mb()`, `time.time()` → **O(1)**


### Summary Table

| Block / Function                                                  | Complexity                  | Notes                                                             |
| ----------------------------------------------------------------- | --------------------------- | ----------------------------------------------------------------- |
| DataLoad / DataCleaner                                            | O(n)                        | Linear traversal of all rows                                      |
| TimeSeries                                                        | O(n)                        | Linear traversal                                                  |
| FeatureExtractor / TemporalFeatureExtractor                       | O(n)                        | Linear per column/row                                             |
| Row-wise classification (`apply`)                                 | O(n)                        | One traversal per column                                          |
| GroupBy aggregations                                              | O(n)                        | Each row visited once per aggregation                             |
| Pairplot                                                          | O(c² * n)                   | Iterates all column pairs and all rows                            |
| Kruskal-Wallis with `groupby().apply(list)`                       | O(n * g) → O(n² worst-case) | Each row may be processed multiple times for groups               |
| Chi-Square (`crosstab`)                                           | O(n * k1 * k2)              | Depends on categorical cardinalities; can exceed O(n)             |
| Multi-boxplot                                                     | O(f * g * n)                | Traverses all rows for each feature and group; can approach O(n²) |
| TableExporter.to_csv / FigureExporter.save                        | O(n)                        | Linear write/render                                               |
| Utility operations (`os.path.join`, `get_memory_mb`, `time.time`) | O(1)                        | Constant-time operations                                          |


### Conclusion

* **Typical complexity**: O(n) for most processing steps.
* **Potential superlinear segments**: groupby + apply, crosstab, pairplot, multi-boxplot.
* **Constant time utilities**: O(1)
  
This analysis highlights the segments that could dominate runtime for very large datasets.

### Performance Analysis

| Metric           | Value        |
| ---------------- | ------------ |
| Total Time       | 76.61 s      |
| Used Memory      | 405.38 MB    |

The execution of the `main()` function took approximately **76.61 seconds** and consumed around **405.38 MB** of memory. This aligns with the theoretical analysis of the pipeline: the overall **time complexity is primarily O(n)**, as most operations traverse the dataset linearly, including feature extraction, classification, and visualization. Memory usage is also roughly proportional to the size of the dataset (**O(n)** in space), since intermediate DataFrames, feature matrices, and plot objects are stored in memory throughout the computation. The observed performance confirms that the pipeline is efficient and scalable for medium-sized datasets, though very large datasets could lead to increased runtime or memory consumption, particularly during multi-feature visualizations and grouped statistical tests.

